In [18]:
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, AncillaRegister, ClassicalRegister
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator

In [23]:
def create_wavelet_registers(
    wavelet_idx,
    n_bits=4,
    postfix=""
):
    return {
        "t0": QuantumRegister(n_bits, name=f"qreg_w{wavelet_idx}_t0_{postfix}"),      # Central time
        "f0": QuantumRegister(n_bits, name=f"qreg_w{wavelet_idx}_f0_{postfix}"),      # Central frequency
        "Q": QuantumRegister(n_bits, name=f"qreg_w{wavelet_idx}_Q_{postfix}"),        # Quality factor
        "A": QuantumRegister(n_bits, name=f"qreg_w{wavelet_idx}_A_{postfix}"),        # Amplitude
        "phi0": QuantumRegister(n_bits, name=f"qreg_w{wavelet_idx}_phi0_{postfix}"),  # Initial phase
    }

def quantum_likelihood_oracle(
    wavelet_regs, qreg_residual, qreg_likelihood_encoding, 
    obs_duration, noise_psd
):
    # @TODO - make more rigorous: exp(-2 * r^2 / (T * S_n))
    
    # Create sub-circuit for evaluation
    all_regs = []
    for w_reg in wavelet_regs:
        all_regs.extend(w_reg.values())
    all_regs.extend([qreg_residual, qreg_likelihood_encoding])
    
    qc = QuantumCircuit(*all_regs)
    
    # 1. Compute residual
    qc.cx(all_regs[0], qreg_residual[0]) # @TODO - make more rigorous
    
    # 2. Compute the square of the residual: |r> -> |r^2> 
    for i in range(qreg_residual.size):
        # Scale rotation angle to match the physical constant: 2 / (T * S_n)
        theta_rotation = 2.0 / (obs_duration * np.mean(noise_psd))
        qc.cry(theta_rotation, qreg_residual[i], qreg_likelihood_encoding[0])
        
    return qc

def quantum_adaptive_proposal_t2(
    current_w, history_w, qreg_proposal_drive, delta=0.25
):
    all_regs = []
    for cw, hw in zip(current_w, history_w):
        all_regs.extend(cw.values())
        all_regs.extend(hw.values())
    all_regs.append(qreg_proposal_drive)
    
    qc = QuantumCircuit(*all_regs)
    
    # Iterate through all current and historical parameters
    for cw, hw in zip(current_w, history_w):
        for param in ["t0", "f0", "Q", "A", "phi0"]:
            curr_reg = cw[param]
            hist_reg = hw[param]
            
            for i in range(curr_reg.size):
                # @TODO - make more rigorous
                
                # Encode implicit quantum covariance term (C + epsilon * I) via entanglement
                qc.cx(hist_reg[i], curr_reg[i])

                # Apply the Gaussian perturbation drive
                qc.cry(delta * np.pi, qreg_proposal_drive[0], curr_reg[i])
                
    return qc

def generate_synthetic_ligo_data(obs_duration=4.0, sample_rate=512, seed=42):
    np.random.seed(seed)
    n_samples = int(obs_duration * sample_rate)
    times = np.linspace(0, obs_duration, n_samples)
    
    # 1. Generate Simulated Noise Power Spectral Density (S_n) and colored noise
    # Standard LIGO-like design sensitivity curve approximation
    freqs = np.fft.rfftfreq(n_samples, d=1/sample_rate)
    noise_psd = 1e-46 * (1.0 + (freqs - 100)**2 / 5000) 
    noise_psd[freqs < 20] = 1e-35 # High low-frequency seismic wall
    
    # Generate colored Gaussian noise
    white_noise = np.random.normal(0, 1, n_samples)
    noise_fft = np.fft.rfft(white_noise) * np.sqrt(noise_psd * n_samples / 2.0)
    noise = np.fft.irfft(noise_fft, n=n_samples)
    
    # 2. Inject a True Signal Wavelet (Target Parameters)
    # t0 = 2.0s, f0 = 80Hz, Q = 8, A = 5e-21, phi0 = pi/4
    t0_true, f0_true, Q_true, A_true, phi0_true = 2.0, 80.0, 8.0, 5e-21, np.pi/4
    tau_true = Q_true / (2 * np.pi * f0_true)
    
    signal = A_true * np.exp(-((times - t0_true) / tau_true)**2) * np.cos(2 * np.pi * f0_true * (times - t0_true) + phi0_true)
    
    # Combined strain data (d = n + s)
    data = noise + signal
    
    return times, data, noise_psd

def fully_quantum_bayeswave_mcmc(
    n_wavelets_max=2,
    n_bits=4,
    obs_duration=4.0
):
    # 0. Setup
    # @TODO - get real data
    times, classical_data, noise_psd = generate_synthetic_ligo_data(obs_duration=obs_duration)
    
    noise_psd = np.array([1.2e-46, 1.5e-46, 1.1e-46]) 
    
    # 0a. Parallel multi-chain
    # Chain 0 regs
    c0_wavelets = [create_wavelet_registers(i, n_bits, "c0") for i in range(n_wavelets_max)]
    
    # Chain 1 regs
    c1_wavelets = [create_wavelet_registers(i, n_bits, "c1") for i in range(n_wavelets_max)]
    
    # History regs
    hist_wavelets = [create_wavelet_registers(i, n_bits, "hist") for i in range(n_wavelets_max)]
    
    # 2. Ancillary regs
    qreg_residual = AncillaRegister(n_bits, name="qreg_residual")
    qreg_lik_encoding = AncillaRegister(1, name="qreg_lik_encoding")
    qreg_swap_accept = AncillaRegister(1, name="qreg_swap_accept")
    
    # Measurement regs
    total_meas_qubits = n_wavelets_max * n_bits * 5
    creg_meas = ClassicalRegister(total_meas_qubits, name="creg_detector_readout")
    
    # Flatten list
    flat_regs = []
    for w in c0_wavelets + c1_wavelets + hist_wavelets:
        for r in w.values():
            flat_regs.append(r)
    flat_regs.extend([qreg_residual, qreg_lik_encoding, qreg_swap_accept])
    
    qc = QuantumCircuit(*flat_regs, creg_meas)

    # State prep
    target_size = 2**n_bits
    step = len(classical_data) // target_size
    cropped_data = classical_data[::step][:target_size]
    norm_factor = np.linalg.norm(cropped_data)
    statevector = cropped_data / norm_factor if norm_factor > 0 else np.zeros(target_size)
    
    qc.initialize(statevector, qreg_residual[:])
    qc.barrier()
    
    # Adaptive proposal
    qc_prop = quantum_adaptive_proposal_t2(
        c0_wavelets, hist_wavelets, qreg_lik_encoding, delta=0.25
    )
    qc.compose(qc_prop, inplace=True)
    
    # Quantum acceptance ratio
    qc_lik = quantum_likelihood_oracle(
        c0_wavelets, qreg_residual, qreg_lik_encoding, obs_duration, noise_psd
    )
    qc.compose(qc_lik, inplace=True)
    
    # 3. APT swaps
    # Put swap ancilla into superposition
    qc.h(qreg_swap_accept[0])
    
    # Swap accepted parameter values between two chains
    for w0, w1 in zip(c0_wavelets, c1_wavelets):
        for param in ["t0", "f0", "Q", "A", "phi0"]:
            for qubit_idx in range(n_bits):
                qc.cswap(qreg_swap_accept[0], w0[param][qubit_idx], w1[param][qubit_idx])
                
    # Measure chain 0 parameter values (final targe chain)
    meas_counter = 0
    for w in c0_wavelets:
        for param in ["t0", "f0", "Q", "A", "phi0"]:
            for q in w[param]:
                qc.measure(q, creg_meas[meas_counter])
                meas_counter += 1
                
    return qc

In [24]:
qc = fully_quantum_bayeswave_mcmc(n_wavelets_max=5, n_bits=3)
print(f"Successfully integrated modules!")
print(f"Total Circuit Qubits: {qc.num_qubits}")
print(f"Total Operations:     {len(qc.data)}")

# Run a quick check using the local simulator
simulator = AerSimulator()
result = simulator.run(qc, shots=10).result()
counts = result.get_counts()
print("\nSample Output Counts from fully integrated simulation:\n", counts)

Simulation failed and returned the following error message:
ERROR:  [Experiment 0] Insufficient memory to run circuit circuit-191 using the statevector simulator. Required memory: 18446744073709551615M, max memory: 15699M


Successfully integrated modules!
Total Circuit Qubits: 230
Total Operations:     309


QiskitError: 'ERROR:  [Experiment 0] Insufficient memory to run circuit circuit-191 using the statevector simulator. Required memory: 18446744073709551615M, max memory: 15699M ,  ERROR: Insufficient memory to run circuit circuit-191 using the statevector simulator. Required memory: 18446744073709551615M, max memory: 15699M'